> **This is `16_shap_models.ipynb` executed under the `lender` run config.**
> Identical code, one line changed: `CFG = train.LENDER`. That swaps the matrix to
> `data/processed/model_matrix_lender/`, the feature list to the 54 columns of
> `targets.FEATURE_COLS_LENDER` (the 41 baseline features plus 13 from the Charges API
> harvest), and adds `primary_lender_group` to the categoricals. Same four targets, same
> three models, same splits as `refactor` (I checked the origin grids match before running).
> `switching` is present in the matrix on purpose and deliberately not trained here.
> Results land in `reports/runs/lender/`, and the A/B against `refactor` is
> `train.compare_runs(["baseline", "refactor", "lender"])`.


# 16 - Step 6: models over the labels, and SHAP

Notebook 15 was step 5: it decided what "good" means. Four self-labelled targets, all standing in an
origin month `t` and looking forward. This notebook is step 6: try to predict them, measure whether it
worked, explain the models with SHAP, and score the live month into an actual ranked list.

**Why I am running this now rather than waiting.** There is a Charges API harvest running on my machine
that will add lender identity (which bank holds each charge, roughly 7.8% coverage so far) and, with it,
the real switching/attrition target that the bulk file cannot see. It is tempting to wait for it. But
training on today's features gives a baseline, and that baseline is the only thing that can ever tell me
whether the harvest was worth twenty-four hours of wall clock. Out-of-time AUC before versus after is a
clean measurement. Skip the baseline and the harvest becomes an act of faith. It is the same A/B design
I already set up for strict versus extended contracts.

So everything below is written to make that later re-run nearly free:

- the feature list is `targets.FEATURE_COLS`, never a copy of it, so new columns arrive by themselves;
- I iterate `targets.TARGETS` rather than writing four literals, so a fifth target appears on its own;
- categoricals go through `targets.CATEGORICAL_COLS` (the lender work adds at least one, `primary_lender_group`);
- the origin grid for each target is read **off that target's own partitions on disk**, not from a shared
  assumption, because a switching target will sit on a different base population ("has an outstanding
  Lloyds charge at `t`") that is much smaller than the "Active at `t`" the current four share.

And one thing I am deliberately *not* doing: **no hyperparameter tuning**. Tuning against a feature set
that is about to gain columns is wasted effort. Tune once, at the end, on the final matrix.

---

## Update, 27 July 2026: I rewrote the training pipeline after the supervisor call

Everything below this box was written and run *before* the call with Fernando. The results are
unchanged and still valid, but the code that produces them has been reorganised, so read this first
or the code cells will not match what I described further down.

**What Fernando asked for**, and what I did about each point:

1. *"You should have a scikit-learn reusable pipeline to train the model."* The statistics were
   already scikit-learn and already correct: the logistic model was a proper `Pipeline` with a
   `ColumnTransformer`, the splits used `GroupShuffleSplit` and `GroupKFold`, and LightGBM was
   already going through its scikit-learn wrapper. What was **not** reusable was the shape. There
   were two hard-coded functions, `fit_lightgbm` and `fit_logistic`, called by name inside
   `run_target`, so adding a model meant editing `run_target`.
2. *"Adding a different model type should be effortless, one line of code."* `src/models/train.py`
   now has a `MODELS` registry. Each entry is a `ModelSpec` that knows how to build a complete
   estimator, whether it needs the dense preprocessing, which SHAP explainer applies to it, and
   what its hyperparameter grid is. `run_target` loops over the registry. Adding a random forest or
   an SVM is genuinely one dict entry.
3. *"Have a set of easy-to-interpret models and, most importantly, a neural network."* I added
   `mlp` (scikit-learn's `MLPClassifier`) to the registry. I also pulled the impute / scale /
   one-hot block out of `fit_logistic` into a shared `dense_preprocessor`, because if the neural
   network and the logistic floor saw differently prepared features then the comparison between
   them would be about the preprocessing rather than about the model family.
4. *"Where a model has hyperparameters, validate more."* There is now `train.tune_model`, and it
   does **not** use a default `KFold`. A plain `KFold` inside a search would reintroduce exactly
   the two leaks (company and time) that the whole out-of-time split exists to prevent, and it
   would pick parameters that only look good because they memorised companies. So the search gets
   `train.OriginEmbargoSplit`, which applies the same embargo rule recursively inside the training
   origins and never touches the test origins.
5. *"SHAP is more useful for complex models."* Worth being precise about this one, because it is
   the heart of the dissertation argument. My design was already right: SHAP only ever ran on
   LightGBM, and the logistic model was read through its raw coefficients, never through SHAP. But
   the reason is sharper than "SHAP is for neural networks". For a linear model the SHAP value of
   feature *j* is just `coef_j * (x_j - E[x_j])`, so it is a rescaling of a number I can already
   read off the model. For LightGBM there is no such number, and `TreeExplainer` recovers one
   exactly and in seconds. For the MLP there is no shortcut at all, so it falls back to
   `KernelExplainer`, which is sampled, approximate, and roughly a thousand times slower. That
   spread is a result in itself and I measure it below.

**What deliberately did not change:** the out-of-time split, the embargo, the unsampled evaluation
population, and the odds recalibration. Those are the parts that are hard to get right and they are
model-agnostic, so a model swap costs none of them. I checked this the boring way: re-running
insolvency through the new code reproduces the old ROC-AUC (0.761520), PR-AUC (0.016923) and
precision@100 (19.0%) to every digit. The `baseline` run in `reports/runs/`
therefore stays a valid reference.

**Stored outputs below are from the pre-refactor run** (LightGBM and logistic only). The cells that
changed are marked. Re-run the notebook top to bottom to get the three-model comparison.

---

## Setup

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
# Every default path in src/ is repo-root relative, so work from there.
os.chdir(REPO_ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from src.features import contracts, panel
from src.models import targets, train

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

# One run = one RunConfig. Everything below reads CFG, so re-running this whole
# notebook against the lender matrix is this line and nothing else:
#     CFG = train.LENDER
CFG = train.LENDER

print(f"run tag: {CFG.tag}")
print(f"matrix : {CFG.matrix_dir}")
print(f"{len(CFG.cols())} features, {len(CFG.target_names)} targets, models {CFG.model_names()}")
print(f"categoricals: {CFG.cats()}")

## The split, and why it is not `train_test_split`

Sneha's baseline notebook uses `train_test_split(X, y, test_size=0.25, stratify=y)`. That is the right
call for the cross-sectional problem she set up and the wrong one here, in two independent ways.

**Company leakage.** Each company appears in up to 33 monthly rows and at every quarterly origin. A random
split puts some of those rows in train and the rest in test, so the model can memorise the company instead
of learning the pattern. The test score comes out flattering and means nothing.

**Time leakage.** A random split mixes 2026 rows into training while testing on 2024. The model gets to
see the future, which in production it never does.

Either one alone invalidates the evaluation. So: **train on early origins, test on late ones**, with an
**embargo** in between.

The embargo is the part that is easy to skip. A training row at origin `t` carries a label that resolves
over `t+1 ... t+H`. If the first test origin sits inside that window, the model was trained on the answer
to its own exam. So I drop every origin where `first_test - t < H` months. At the boundary, where
`first_test - t == H` exactly, the training label resolves in the test origin's *feature* month, which is
data the model is handed anyway, and the test labels only start at `first_test + 1`. That is clean, so the
rule is a gap of **at least** `H`.

In [ ]:
splits = {t: train.split_origins(t, CFG) for t in CFG.target_names}
for s in splits.values():
    print(s.describe())
    print()

Three things to read off that.

**Lending loses nothing to the embargo.** Its horizon is 3 months and the origins are quarterly, so
consecutive origins are already exactly `H` apart. That was the point of quarterly origins in step 5, and
it is why lending gets five training origins where insolvency gets three.

**Growth had to give up the early-origin filter.** `targets.FIRST_FULL_ORIGIN` (2024-10) is the first
month where the 12-month deltas are actually formed rather than NULL for calendar reasons. I would rather
train past it, because that missingness pattern lands entirely in train and never in test, and it is
exactly the sort of artefact a tree will happily split on and then be unable to use. But growth has a
12-month horizon, so the embargo eats three whole origins, and honouring the filter as well would leave it
with none. `split_origins` drops the filter automatically when it would leave fewer than three training
origins, and records that it did. Growth therefore trains on 2023-10 to 2024-04 where the 12m deltas are
partly NULL. Worth remembering when reading its numbers.

**Voluntary exit and insolvency share a grid but not a difficulty.** Same horizon, same origins, wildly
different base rates (about 7.5% against about 0.33%).

## precision@N has to be measured on the unsampled population

This is the subtlest thing in the notebook and it took me a second pass to get right.

Step 5 wrote the modelling matrix with **negatives downsampled to roughly 10x the positives**, which is
what makes a 0.33% label trainable at all. But that matrix is no longer the population the model would
actually rank. If I compute precision@100 on it, I am asking "of the top 100 out of a pile I already
enriched 30-fold with positives, how many were right", and the answer is inflated by roughly that factor.

ROC-AUC survives negative downsampling, because it only compares positives against negatives pairwise and
downsampling negatives uniformly does not change that ordering in expectation. **PR-AUC and precision@N do
not survive it**, and precision@N is the headline metric here: a relationship manager works a finite call
list, so "of our top 500 picks, how many were right" is the question that matters. AUC over 1.4 million
companies does not answer it.

So the test origins get rebuilt with no downsampling at all. `targets.build_matrix` caps the negative keep
rate at 1.0, so a large enough `neg_ratio` simply keeps everything, and I did not have to write a second
query to get there.

In [ ]:
eval_summary = pd.concat([
    train.build_eval_matrix(t, splits[t].test, CFG, threads=6)
    for t in CFG.target_names
], ignore_index=True)
eval_summary

`sampled_rate` there is now the **true** base rate, and every one of them lands where notebook 15 measured
it: lending 0.26-0.28%, insolvency 0.33%, voluntary exit 7.2-8.2%, growth 2.1%. That is the same base-rate
assertion from step 5, arriving a second time by a different route, which is a reassuring thing for it to do.

## Train

**Changed by the 27 Jul refactor.** This section used to fit exactly two models, named in the body of
`run_target`. It now loops over the `train.MODELS` registry, which is what makes the model comparison
Fernando asked for a one-line change rather than an edit.

Three models per target now.

**LightGBM** is the real model. **Logistic regression** is the interpretable floor, and it is worth the
extra twenty minutes for two reasons: if the boosting cannot clearly beat a regularised linear model on
the same features then it is not earning its complexity and I should say so, and its signed coefficients
are an independent check on the SHAP directions. If SHAP says more charges means more distress and the
coefficient says the opposite, one of them is wrong.

**The MLP is new.** A small feed-forward neural network, `hidden_layer_sizes=(64, 32)`, sitting on the
same shared preprocessing as the logistic model. I picked scikit-learn's `MLPClassifier` rather than
torch or keras on purpose: it satisfies the same `fit` / `predict_proba` contract as everything else, so
it drops straight into the registry and costs no new dependency, and for around sixty tabular features
it is the right size of tool. A deeper network would be a claim about the data that I have no evidence
for.

Its job here is not to win. It is to occupy the complex end of the interpretability/accuracy trade-off
that the dissertation argues about. LightGBM is awkward for that argument because `TreeExplainer` gives
it exact SHAP values almost for free, so it is complex *and* cheaply explainable. The MLP is the honest
illustration of the trade-off: no readable parameters, and no shortcut for recovering an explanation
afterwards either.

Four implementation notes.

**Early stopping uses a company-grouped slice of the training rows, never the test set.** Stopping on the
test set is tuning on the exam, just more politely. Grouped rather than random for the same leakage reason
as the main split. This applies to LightGBM; `MLPClassifier` does its own early stopping on a *random*
slice and scikit-learn gives me no way to hand it a grouped one, so stopping may run slightly late there.
That affects only when training halts, never the out-of-time evaluation, and I would rather write the
limitation down than pretend it away.

**Dense models are row-capped.** LightGBM eats everything; the logistic model caps at 500k rows and the
MLP at 300k. Backprop over a dense one-hot matrix is by far the slowest thing in the module and the
number I want from it is a fair comparison, not the best obtainable MLP.

**Missingness is flagged, not filled.** `SimpleImputer(add_indicator=True)` in the shared preprocessor.
A NULL delta around the June 2025 panel hole means "cannot say" and that is information. LightGBM can
express that natively; the dense models need the indicator column to say it at all.

**`n_jobs=6`, not `-1`.** This is not a tuning knob, it is a workaround, and it cost me an hour so it is
going in the notebook. Under WSL2 `n_jobs=-1` reads 22 logical cores and then spends essentially all its
time in thread contention: 50 boosting rounds on the 165k-row insolvency matrix took **98 seconds** at
`n_jobs=-1` and **0.9 seconds** at `n_jobs=8`. Two orders of magnitude, identical result.

In [ ]:
# The registry is the whole point of the refactor: this is the list run_target loops
# over, and adding to it is the one-line change.
for name, spec in train.MODELS.items():
    print(f"{name:<9} explainer={spec.explainer:<7} max_rows={spec.max_rows} "
          f"early_stopping={spec.early_stopping}")

In [ ]:
%%time
# CFG carries the matrix, the feature list, the categoricals and the model set, so
# they cannot drift apart. For one model only:
#     from dataclasses import replace; replace(CFG, models=("lightgbm",))
results = {t: train.run_target(t, CFG) for t in CFG.target_names}

In [ ]:
overview = pd.DataFrame([
    {
        "target": r["target"],
        "H (m)": r["horizon_m"],
        "train rows": r["train_rows"],
        "train pos": r["train_positives"],
        "test rows": r["metrics"]["lightgbm"]["n"],
        "test base rate": r["metrics"]["lightgbm"]["base_rate"],
        "best iter": r["best_iteration"],
    }
    for r in results.values()
])
overview.style.format({"train rows": "{:,}", "train pos": "{:,}",
                       "test rows": "{:,}", "test base rate": "{:.3%}"})

## Out-of-time results

In [ ]:
rows = []
for r in results.values():
    for model, m in r["metrics"].items():
        rows.append({"target": r["target"], "model": model, "base_rate": m["base_rate"],
                     "roc_auc": m["roc_auc"], "pr_auc": m["pr_auc"],
                     **{f"P@{n}": m[f"precision_at_{n}"] for n in train.TOP_N},
                     **{f"lift@{n}": m[f"lift_at_{n}"] for n in train.TOP_N}})
scores = pd.DataFrame(rows)
scores.style.format({"base_rate": "{:.3%}", "roc_auc": "{:.4f}", "pr_auc": "{:.4f}",
                     **{f"P@{n}": "{:.1%}" for n in train.TOP_N},
                     **{f"lift@{n}": "{:.0f}x" for n in train.TOP_N}})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
piv_auc = scores.pivot(index="target", columns="model", values="roc_auc")
piv_auc.plot.bar(ax=axes[0], rot=0, width=0.75)
axes[0].axhline(0.5, color="grey", ls="--", lw=1)
axes[0].set_ylim(0.45, 1.0)
axes[0].set_ylabel("out-of-time ROC-AUC")
axes[0].set_title("LightGBM against the logistic floor")

piv_lift = scores[scores.model == "lightgbm"].set_index("target")[[f"lift@{n}" for n in train.TOP_N]]
piv_lift.plot.bar(ax=axes[1], rot=0, width=0.75, logy=True)
axes[1].axhline(1, color="grey", ls="--", lw=1)
axes[1].set_ylabel("lift over base rate (log scale)")
axes[1].set_title("How much better than calling at random")
plt.tight_layout()
plt.show()

### The trade-off, which is the thing Fernando wants argued in the dissertation

The expectation going in is the standard one: the complex model wins on accuracy and loses on
interpretability, and the simple model does the reverse. What the three-model table above lets me say is
whether that expectation actually holds *on this data*, and it is more interesting than a straight yes.

The order I get on insolvency is logistic < MLP < LightGBM. So the neural network does buy something over
the linear floor, which says there is genuine non-linearity in these features and the floor is leaving it
on the table. But it does not catch LightGBM. That is not surprising for tabular data with this shape:
gradient boosted trees handle mixed types, missing values and sharp thresholds natively, and a dense
network has to learn all three from scratch through a one-hot layer.

Which gives the dissertation a more honest story than "complex beats simple". The winner here is the model
that is complex *in the right way for the data*, and its explanations happen to be exact and cheap. The
neural network is the one that pays the full interpretability price without collecting the full accuracy
reward. That is worth arguing, and it is only arguable because the registry made fitting a third family
cost one line.

## Recalibration

Downsampling negatives at rate `r` multiplies the odds by `1/r`, so every predicted probability comes out
inflated by roughly 30x. Ranking survives that perfectly, because it is a monotone transform. The numbers
do not, and "this company has a 12% chance of insolvency" is a lie until it is corrected.

The correction is one line: `odds_true = odds_sampled * r`. Step 5 carried `neg_keep_rate` along on every
row precisely so step 6 could undo its own sampling without having to remember anything.

The check below is that the mean recalibrated prediction lands on the true base rate of the test
population. It is the cheapest possible test that the correction is the right way round, and getting it
backwards would be an easy mistake to not notice.

In [ ]:
# Changed by the refactor: calibration is now recorded per model, because every model
# trained on the same downsampled matrix and so every model needs the same correction.
calib = pd.DataFrame([
    {"target": t, "model": m,
     "train_neg_keep_rate": r["calibration"]["train_neg_keep_rate"],
     "test_base_rate": r["calibration"]["test_base_rate"], **c}
    for t, r in results.items()
    for m, c in r["calibration"]["models"].items()
])
calib["raw_inflation"] = calib["mean_pred_raw"] / calib["test_base_rate"]
calib["recal_ratio"] = calib["mean_pred_recalibrated"] / calib["test_base_rate"]
calib.style.format({"train_neg_keep_rate": "{:.4f}", "mean_pred_raw": "{:.4%}",
                    "mean_pred_recalibrated": "{:.4%}", "test_base_rate": "{:.4%}",
                    "raw_inflation": "{:.1f}x", "recal_ratio": "{:.2f}x"})

## Secondary check: GroupKFold by company

Out-of-time is the primary split because it mirrors production. Grouped CV ignores time on purpose, and the
comparison between the two is the informative part: if grouped CV is much the better of the two, the model
has learned something about this particular period that will not survive into the next one. If they agree,
it generalises.

Row-capped at two million, because the point is a comparison and not a better estimate, and the
voluntary-exit matrix is twelve million rows.

In [ ]:
%%time
cv = pd.DataFrame([train.grouped_cv_auc(t, CFG) for t in CFG.target_names])
cv["oot_roc_auc"] = cv["target"].map(lambda t: results[t]["metrics"]["lightgbm"]["roc_auc"])
cv["gap"] = cv["mean_roc_auc"] - cv["oot_roc_auc"]
cv[["target", "rows", "mean_roc_auc", "oot_roc_auc", "gap"]].style.format(
    {"rows": "{:,}", "mean_roc_auc": "{:.4f}", "oot_roc_auc": "{:.4f}", "gap": "{:+.4f}"})

## Hyperparameter validation, and why it cannot be `GridSearchCV(cv=5)`

This is Fernando's fourth point and it is the one with a trap in it.

Everything above is untuned on purpose. Tuning against a feature set that is about to gain the lender
columns is wasted effort, so the plan was always to tune once at the end on the final matrix. But the
machinery has to exist and be correct before then, and the correct version is not the obvious one.

The obvious version is `GridSearchCV(estimator, grid, cv=5)`. That would be wrong here for exactly the
reasons the main split is not `train_test_split`. The default `KFold` shuffles rows, so a company that
appears at five origins lands on both sides of every fold boundary, and 2026 rows end up training a model
validated on 2024 rows. The search would then pick the parameters that memorise companies best, and I
would have leaked at the tuning stage after being careful about it at the training stage. Worse, it would
look fine: the reported CV score would be high and the out-of-time score would quietly not improve.

So `train.tune_model` uses `train.OriginEmbargoSplit`, which applies the primary split's rule recursively
*inside the training origins only*. Fold k trains on every origin at least H months before validation
origin k and validates on that origin alone. The test origins are never touched by tuning.

One consequence worth stating rather than hiding. `n_splits` is a maximum, not a promise. Insolvency has
three training origins and a six-month horizon, so exactly one validation origin has anything legal left
to train on: **one fold**. That is not a defect in the splitter, it is the real amount of independent
evidence a 33-month panel with quarterly origins can offer for tuning a six-month label. A `KFold` would
have cheerfully invented five folds by leaking. Lending, with a three-month horizon and five training
origins, gets the full three.

In [ ]:
%%time
# Small n_iter here because this is a demonstration that the machinery is correct, not
# the final tuning run. Note the fold counts: lending 3, insolvency 1, for the reason above.
tuning = pd.DataFrame([
    {k: v for k, v in train.tune_model(m, t, CFG, n_iter=4).items() if k != "cv_results"}
    for t in ("lending", "insolvency")
    for m in ("logistic", "lightgbm")
])
tuning

`C=1.0` comes out on top for the logistic model on both targets, which is the value the baseline already
used, so the floor was not being held back by its regularisation. LightGBM prefers more leaves and a
slower learning rate than the fixed settings, which is the expected direction and is worth a real search
once the feature set stops moving.

## SHAP

**Changed by the refactor.** This used to call `shap.TreeExplainer` directly. It now goes through
`train.explain`, which dispatches on the model family, because with three model families in the registry
"which explainer" is no longer a question with one answer. That dispatch is where Fernando's point about
SHAP and complex models turns into something I can measure rather than assert:

- **`tree`** (LightGBM): `TreeExplainer`, and it is **exact** for tree models rather than an
  approximation. 100k rows in seconds.
- **`linear`** (logistic): available, and almost pointless. For a linear model the SHAP value of feature
  *j* is `coef_j * (x_j - E[x_j])`, a rescaling of a coefficient I can already read straight off the
  model. This is precisely Fernando's argument for why a simple model does not need SHAP, and it is why
  the direction check further down reads the logistic model's coefficients rather than explaining it.
- **`kernel`** (MLP): model-agnostic sampling, approximate, and slow. Measured below.

The main run is on LightGBM, on a stratified 100k subsample of the test population. Stratified matters
here: a plain 100k draw from a 0.33% base rate holds about 330 positives, and the summary plot would end
up describing the negatives almost exclusively.

A note on why SHAP and not LIME, since it came up. LIME is not a model or a training framework, it is a
post-hoc explainer, the same category of tool as SHAP; you do not "train on LIME". And for trees SHAP is
strictly better: exact rather than a local surrogate, stable across re-runs rather than resampled (which
matters a lot when the explanation is going in front of a relationship manager), and additive, so the
values sum to the prediction. That last property is what lets a composite index be decomposed into
contributions at all.

In [ ]:
%%time
explanations = {}
for t, r in results.items():
    Xs, ys = train.shap_sample(r["_X_test"], r["_y_test"])
    # train.explain picks the explainer off the model's ModelSpec. For lightgbm that is
    # TreeExplainer, exact, and it also normalises the two shapes LightGBM can return
    # for a binary problem depending on version.
    sv = train.explain("lightgbm", r["_models"]["lightgbm"], Xs)
    explanations[t] = {"X": Xs, "y": ys, "shap": sv,
                       "importance": train.shap_importance(sv, Xs)}
    print(f"{t}: {sv.shape[0]:,} rows x {sv.shape[1]} features")

In [ ]:
top = pd.concat([
    e["importance"].head(10).assign(target=t, rank=range(1, 11))
    for t, e in explanations.items()
])
top.pivot(index="rank", columns="target", values="feature")

In [ ]:
for t, e in explanations.items():
    shap.summary_plot(e["shap"], e["X"], max_display=15, show=False)
    plt.title(f"{t} (H={targets.ALL_TARGETS[t][1]}m)", loc="left")
    plt.tight_layout()
    plt.show()

### What explaining the neural network actually costs

The cell above explained 100k rows of a boosted ensemble exactly, in seconds. Here is the same question
asked of the MLP, on twenty rows, to show why I do not run it on 100k.

`KernelExplainer` has no access to the model's internals. It repeatedly perturbs the input, re-runs
`predict_proba`, and fits a weighted local linear model around each row, so its cost scales with rows
times samples times a forward pass. It is also an approximation, and re-running it with a different seed
gives slightly different numbers.

Put the three together and the trade-off Fernando wants argued has a concrete price tag attached:

| model | explanation | exact? | 100k rows |
|---|---|---|---|
| logistic | read `coef_` | exact, and free | instant |
| LightGBM | `TreeExplainer` | exact | seconds |
| MLP | `KernelExplainer` | sampled approximation | hours |

That middle row is the reason the boosted model is the one going into production. It is not simply that
it scores best out of time. It is that it scores best out of time *and* an RM can be handed the three
reasons behind any individual company's score without a compute budget.

In [ ]:
%%time
# Twenty rows, not 100k. At the measured rate 100k rows would take several hours.
t = "insolvency"
e = explanations[t]
sv_mlp = train.explain("mlp", results[t]["_models"]["mlp"], e["X"].head(20),
                       background=e["X"], nsamples=100)
print(f"kernel SHAP on {sv_mlp.shape[0]} rows x {sv_mlp.shape[1]} features")
train.shap_importance(sv_mlp, e["X"].head(20)).head(8)

### Does the logistic floor agree with SHAP on direction?

The point of fitting the linear model was not its AUC, it was this. Two independent views of the same
question: does feature X push the prediction up or down. Where they disagree, one of them is wrong and it
is worth knowing which before any of this reaches a relationship manager.

One thing I got wrong on the first pass and am writing down so nobody repeats it: **the mean signed SHAP
value is not a direction.** It is the average contribution across the sample, and for a feature whose
effect is non-monotone (say, both very new and very old companies borrow more) it can average to nearly
anything, including the opposite of the truth. The direction of a feature is the relationship between the
**feature's value and its own SHAP value**, which is what the colour gradient in a summary plot shows. So
that is what I measure, as a Spearman correlation, which does not assume the relationship is linear.

Categoricals are skipped here. `sector` and `segment` one-hot into many coefficients and do not have a
single direction to compare, so asking the question of them is not meaningful.

In [ ]:
from scipy.stats import spearmanr


def logit_directions(target, k=12):
    e = explanations[target]
    pipe = results[target]["_models"]["logistic"]
    names = pipe.named_steps["pre"].get_feature_names_out()
    coefs = pd.Series(pipe.named_steps["clf"].coef_[0], index=names)
    cols = list(e["X"].columns)
    out = []
    for row in e["importance"].head(k).itertuples():
        # Drop the imputer's missingness indicators: "was this NULL" is a different
        # question from "which way does the value push", and matching on the suffix
        # alone would happily pick the indicator's coefficient instead. Categoricals
        # match nothing here and fall out, which is the intent.
        hits = coefs[[n for n in names
                      if n.endswith("__" + row.feature) and "missingindicator" not in n]]
        if len(hits) == 0:
            continue
        j = cols.index(row.feature)
        x = e["X"].iloc[:, j].to_numpy(dtype=float)
        ok = ~np.isnan(x)
        if ok.sum() < 1000 or np.nanstd(x[ok]) == 0:
            continue
        rho = float(spearmanr(x[ok], e["shap"][ok, j]).statistic)
        c = float(hits.iloc[int(np.argmax(np.abs(hits.to_numpy())))])
        out.append({"feature": row.feature, "shap_direction": rho, "logit_coef": c,
                    "agree": np.sign(rho) == np.sign(c)})
    return pd.DataFrame(out).assign(target=target)


directions = pd.concat([logit_directions(t) for t in targets.TARGETS], ignore_index=True)
directions.pivot_table(index="target", columns="agree", values="feature",
                       aggfunc="count").fillna(0).astype(int)

In [ ]:
directions[~directions["agree"]][["target", "feature", "shap_direction", "logit_coef"]]

Disagreements are not automatically a bug. The logistic coefficient is a *partial* effect holding every
other feature fixed, and with features as correlated as these (`d_charges_3m`, `d_charges_6m` and
`d_charges_12m` measure overlapping windows of the same thing) a partial effect can legitimately flip sign
against the marginal relationship SHAP is showing. What the table is for is spotting a feature where the
two disagree *and* the SHAP direction is economically implausible. That is the one to go and look at.

## Score the live month

Score wide and cheap, enrich narrow and expensive. Every active company in the latest panel month gets a
recalibrated probability from each of the four models; only the top few hundred are ever worth spending a
Companies House API call on.

Two caveats that belong on the output rather than in my head.

**The contract features in this month are past the harvest watermark.** The two OCDS bulk files refresh on
different cadences, so contract counts in the final month or two fall because we stopped observing, not
because companies stopped winning work. Fine to score on, not fine to train on, which is why
`targets._check_contract_watermark` refuses the latter.

**These are prospect scores, not decisions.** A high insolvency score is a reason to look, not a reason to
decline.

In [ ]:
%%time
live = train.load_scoring_frame(CFG)
# The live month need not contain every sector/segment level the models saw, and
# category codes are positional. Pin them to the training levels before predicting.
train.apply_categories(live, results["lending"]["_X_test"], CFG.cats())
print(f"{len(live):,} active companies at {panel.LAST_MONTH}")

for t, r in results.items():
    live[f"score_{t}"] = train.score_frame(
        r["_models"]["lightgbm"], live, r["calibration"]["train_neg_keep_rate"], CFG)

live[[f"score_{t}" for t in CFG.target_names]].describe().T.style.format("{:.4%}")

The means sit on the labelled base rates, which is the calibration check surviving contact with a month
the models have never seen. That is about as good a smoke test of the whole pipeline as I have.

In [ ]:
def top_prospects(target, n=10):
    cols = ["CompanyNumber", "CompanyName", f"score_{target}", "segment", "sector",
            "Mortgages.NumMortCharges", "new_charge_events_12m", "accounts_overdue_streak_months"]
    return (live.nlargest(n, f"score_{target}")[cols]
                .rename(columns={f"score_{target}": "score"})
                .assign(target=target))

pd.concat([top_prospects(t, 5) for t in CFG.target_names], ignore_index=True)

In [ ]:
# Tagged, so the lender run's scores land beside the baseline's rather than on top.
SCORE_PATH = train.score_path(CFG)
train.SCORE_DIR.mkdir(parents=True, exist_ok=True)
live[["CompanyNumber", "CompanyName", "origin_month",
      *[f"score_{t}" for t in CFG.target_names]]].to_parquet(SCORE_PATH, index=False)
print(f"wrote {SCORE_PATH} ({SCORE_PATH.stat().st_size / 1e6:.1f} MB)")

## Record the run

**Restructured 28 Jul 2026.** This used to write into a flat `reports/step6/`: one
`model_results.json` keyed by tag, and SHAP tables whose tag lived in the filename. It worked
for one run and stopped working the moment I had three things I wanted to compare (the plain
matrix, the lender matrix, and eventually a tuned run). Two problems with it. The SHAP tables
did not travel with the metrics they describe, and nothing recorded *what was run* beyond the
feature list, so "which matrix produced this?" was a question for my memory or the git log.

Every run now owns a directory:

```
reports/runs/<tag>/manifest.json                 how it was run
                   metrics.json                  what came out
                   shap_importance_<target>.csv  one per target
reports/runs/index.csv                           one row per run x target x model
```

The **manifest** is the new part and it is the whole point: it records the matrix directory,
the eval directory, the contract and lender sources, the exact feature list plus a short hash
of it, the categoricals, the targets, the model set, the registry as it stood, the LightGBM
parameters, the git commit and the timestamp. A run is reproducible from it without reading
any code, and two runs are comparable by diffing two manifests rather than by remembering.

`train.record_run` **refuses to overwrite an existing tag.** That is deliberate: the `baseline`
run is the fixed reference the whole A/B design depends on, it is the number that says whether
twenty-four hours of Charges API harvesting bought anything, and under the old flat scheme
re-running one cell would have silently destroyed it. Now it raises. The tag comes from `CFG`,
so a new comparison means a new `RunConfig`, which is the same one line that switches the
matrix and the feature list.

`reports/runs/index.csv` and `train.load_runs()` are both regenerated from the directories on
every write, so the comparison table cannot drift from what is actually on disk. The old
`baseline` files were migrated into `reports/runs/baseline/` unchanged; the numbers are the
same bytes, they just live inside their run now.

In [ ]:
# record_run refuses to overwrite an existing tag, so the migrated `baseline` run
# cannot be destroyed by a re-run of this cell. Change CFG's tag, not this call.
run_path = train.record_run(
    list(results.values()),
    CFG,
    grouped_cv=cv.to_dict("records"),
    direction_check=directions.to_dict("records"),
    tuning=tuning.to_dict("records"),
    scores=str(SCORE_PATH),
)
print(run_path)

for t, e in explanations.items():
    out = train.shap_importance_path(t, CFG.tag)
    e["importance"].to_csv(out, index=False)
    print(out)

# The comparison table, rebuilt from the run directories every time.
train.load_runs()[["tag", "target", "model", "n_features", "roc_auc", "precision_at_500"]]

## What I would do next, in order

1. **The lender features.** They landed in notebook 14b: `data/processed/model_matrix_lender/`,
   41 features to 54. The run is `CFG = train.LENDER` at the top of this notebook and nothing
   else, and it records itself under `reports/runs/lender/`. Then
   `train.compare_runs(["baseline", "lender"])` is the answer to whether the harvest bought
   predictive power. Watch `is_lbg_client` in the SHAP plots: if it ranks highly on `lending`
   that is as much a statement about who Lloyds already banks as about the company.
2. **The strict versus extended contract A/B.** Already set up in step 4b: build the matrices
   against `contracts.ASOF_EXT_DIR` and run them under a `contracts_ext` tag. My guess is it
   helps lending and does nothing for insolvency, but that is a prediction to test.
3. **Then hyperparameters.** Tuning against a feature set that is still moving is wasted
   effort. Tune once the lender question is settled, on whichever matrix won, and record it as
   its own tag so the untuned run stays there to be compared against.
4. **The composite indices.** This is the real deliverable: the model scores become **Lending
   Readiness**, **Credit Risk Exposure** and **Growth Signal**, with per-company SHAP reasons
   attached, so the output is "this company, this score, and here are the three things driving
   it" rather than an unexplained number.
5. **Not a fifth model.** The `switching` target from notebook 14b has 797 positives, under the
   threshold the plan set in advance, so it ships as a rule-based feed rather than a model. The
   lender columns are still features here; only the target is on hold.